# 10 - DL Equal-Weight Ensemble

LSTM, 1DCNN, TCN, TST의 날짜별 예측 로그를 결합하여 equal-weight 앙상블을 평가한다.

- `core`, `momentum`, `extended`는 서로 섞지 않는다.
- `static`, `expanding`은 서로 섞지 않는다.
- 동일한 `(regime, country, feature_set, protocol)` 안에서만 모델 예측을 평균낸다.
- 모델 재학습은 하지 않으며, 저장된 예측값의 평균과 metric 계산만 수행한다.
- TCN expanding 로그가 업로드된 뒤 실행해야 네 모델 전체 조합이 완성된다.

## 0. Colab 셋업

In [ ]:
from pathlib import Path

if not Path('/content/FINTEL').exists():
    !git clone -b Ensemble_JH https://github.com/GHLee1016/FINTEL.git /content/FINTEL

%cd /content/FINTEL
!git fetch origin Ensemble_JH
!git checkout Ensemble_JH
!git pull --ff-only origin Ensemble_JH
!pip install -q -r requirements.txt

print('Ensemble run: prediction averaging only (no GPU training required).')

## 1. Imports / 출력 위치

In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ensemble import (
    build_equal_weight_predictions,
    build_single_vs_ensemble_summary,
    collect_prediction_master,
    coverage_table,
    evaluate_ensemble_predictions,
    validate_full_coverage,
)

RESULTS_DIR = PROJECT_ROOT / 'results'
LOG_DIR = PROJECT_ROOT / 'log' / 'Ensemble'
RESULTS_DIR.mkdir(exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RESULTS_DIR =', RESULTS_DIR)
print('LOG_DIR =', LOG_DIR)

## 2. 최신 DL prediction log 수집 및 완전성 확인

각 모델 담당 브랜치에서 날짜별 예측 로그를 직접 읽는다. TCN expanding 로그가 아직 없다면 여기서 누락 cell을 표시하고 중단된다.

In [ ]:
t0 = time.time()
prediction_master = collect_prediction_master()
coverage = coverage_table(prediction_master)

coverage_summary = coverage.pivot_table(
    index=['model', 'protocol'],
    columns='feature_set',
    values='n_rows',
    aggfunc='sum',
)
display(coverage_summary)

missing = coverage[coverage['n_rows'] == 0]
if not missing.empty:
    print('아직 비어 있는 모델/cell:')
    display(missing[['model', 'regime', 'country', 'feature_set', 'protocol']])

validate_full_coverage(prediction_master)
prediction_master.to_csv(LOG_DIR / 'prediction_master_dl.csv', index=False, encoding='utf-8-sig')
print(f'완전성 확인 완료: {len(prediction_master):,} model prediction rows | {time.time() - t0:.1f}s')

## 3. Equal-Weight 앙상블 예측 생성

네 모델의 모든 2-model, 3-model, 4-model 조합을 계산한다. Cell당 11개 조합이며 전체 Full Test 조합 수는 `72 × 11 = 792`개이다.

In [ ]:
t0 = time.time()
ensemble_predictions = build_equal_weight_predictions(prediction_master)
ensemble_predictions.to_csv(LOG_DIR / 'ensemble_prediction_log.csv', index=False, encoding='utf-8-sig')

print(f'ensemble date predictions: {len(ensemble_predictions):,} rows')
print('model combinations:')
display(ensemble_predictions[['models', 'n_models']].drop_duplicates().sort_values(['n_models', 'models']))
print(f'prediction 생성 완료: {time.time() - t0:.1f}s')

## 4. 지표 평가 및 결과 CSV 저장

In [ ]:
t0 = time.time()
ensemble_results = evaluate_ensemble_predictions(ensemble_predictions)
ensemble_results.to_csv(RESULTS_DIR / 'ensemble_results_equal_weight.csv', index=False, encoding='utf-8-sig')

full_test = ensemble_results[ensemble_results['phase'] == 'Full Test']
print(f'all phase result rows: {len(ensemble_results):,}')
print(f'Full Test evaluations: {len(full_test):,} (expected: 792)')
display(full_test.sort_values('QLIKE').head(20))
print(f'metric 평가 완료: {time.time() - t0:.1f}s')

## 5. Best Single DL 대비 Best Ensemble 비교

각 cell에서 `QLIKE` 기준 최우수 단일 DL 모델과 최우수 equal-weight 앙상블을 비교한다.

In [ ]:
comparison = build_single_vs_ensemble_summary(
    RESULTS_DIR / 'dl_results_master.csv',
    ensemble_results,
)
comparison.to_csv(RESULTS_DIR / 'ensemble_best_comparison.csv', index=False, encoding='utf-8-sig')

n_better = int(comparison['ensemble_better_QLIKE'].sum())
print(f'QLIKE 기준 ensemble 개선 cell: {n_better}/{len(comparison)}')
display(comparison.sort_values('QLIKE_improvement', ascending=False).head(20))

## 6. 결과 CSV GitHub push

위 결과 확인 후 실행한다. 입력 원본을 모은 `prediction_master_dl.csv`는 재생성 가능하므로 push하지 않고, 앙상블 결과와 앙상블 날짜별 로그만 저장한다.

In [ ]:
# from getpass import getpass
# token = getpass('GitHub token: ')
# !git remote set-url origin https://{token}@github.com/GHLee1016/FINTEL.git
# !git config user.name 'Ruysong'
# !git config user.email 'rruysong@gmail.com'
# !git add results/ensemble_results_equal_weight.csv results/ensemble_best_comparison.csv
# !git add log/Ensemble/ensemble_prediction_log.csv
# !git commit -m 'Add DL equal-weight ensemble results'
# !git push origin Ensemble_JH